# CIFAR-10 multi-fold transfer study on Apple Silicon

This notebook is the analysis surface for the upgraded experiment: dense confidence-margin rewards, multiple independently initialized source victims, all three leave-one-family-out folds, query-matched controls, and repeated seeds. Training remains package-backed and resumable.

In [ ]:
import json
from pathlib import Path
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'rl_transfer').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the repository')

REPO_ROOT = find_repo_root(Path.cwd())
QUICK_CONFIG = REPO_ROOT / 'configs/rl_transfer/cifar10_m4_study_quick.json'
FULL_CONFIG = REPO_ROOT / 'configs/rl_transfer/cifar10_m4_study.json'
RUN_STUDY = False
USE_FULL_CONFIG = False
DEVICE = 'auto'
print(f'Repository: {REPO_ROOT}')

## Optional resumable training

Start with the one-seed three-fold diagnostic. Promote to the three-seed study only after every victim-quality gate passes.

In [ ]:
selected_config = FULL_CONFIG if USE_FULL_CONFIG else QUICK_CONFIG
if RUN_STUDY:
    subprocess.run(
        [sys.executable, '-m', 'rl_transfer.cifar_study_cli', '--config', str(selected_config), '--device', DEVICE],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print(f'Training disabled. Selected config: {selected_config.relative_to(REPO_ROOT)}')

In [ ]:
study_name = 'cifar10-m4-study' if USE_FULL_CONFIG else 'cifar10-m4-study-quick'
manifest_path = REPO_ROOT / 'output/rl_transfer/cifar10_m4_studies' / study_name / 'study_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'No study manifest yet: {manifest_path}. Set RUN_STUDY=True first.')
study = json.loads(manifest_path.read_text())
print(f"Status: {study['status']}")
print(f"Runs: {len(study['runs'])}")
print(f"Promotion gate: {study['promotion_gate']['passed']}")

## Cross-fold comparison

In [ ]:
for target_family, methods in study['aggregate'].items():
    print(f'\nHeld-out target: {target_family}')
    print(f"{'method':<38} {'mean ASR':>10} {'mean AUC':>10} {'entropy':>10}")
    for method, metrics in methods.items():
        print(
            f"{method:<38} "
            f"{metrics['final_asr']['mean']:>10.3f} "
            f"{metrics['asr_query_auc']['mean']:>10.3f} "
            f"{metrics['action_entropy']['mean']:>10.3f}"
        )

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Install the analysis extra to render plots.")
else:
    learned_name = 'groupdro_recurrent_ppo_stochastic'
    controls = ('random_action', 'bandit_action', 'score_greedy')
    families = list(study['aggregate'])
    positions = list(range(len(families)))
    width = 0.2
    for offset, method in enumerate((learned_name, *controls)):
        values = [study['aggregate'][family][method]['final_asr']['mean'] for family in families]
        plt.bar([position + (offset - 1.5) * width for position in positions], values, width, label=method)
    plt.xticks(positions, families, rotation=15)
    plt.ylabel('Mean ASR at final query budget')
    plt.title('Frozen transfer by held-out victim family')
    plt.legend()
    plt.show()

## Promotion rule

A fold passes only with at least three exactly aligned seeds, passing victim-quality gates, paired Student-t 95% lower bounds above zero for RL minus random, bandit, and score-greedy controls in both final ASR and ASR/query AUC, and non-collapsed action entropy between 0.10 and 0.95.

In [ ]:
print(json.dumps(study['promotion_gate'], indent=2))